# SPARK ETL - BẢNG PHỤ, TỐI ƯU & BENCHMARK

## PHẦN 1: KẾT NỐI SPARK VÀ HDFS

In [3]:
from pyspark.sql import SparkSession, functions as F, Window

BASE = "hdfs://namenode:8020"
spark = (SparkSession.builder.appName("giai-doan-4")
         .config("spark.hadoop.fs.defaultFS", BASE)
         .config("spark.sql.warehouse.dir", f"{BASE}/user/hive/warehouse")
         .config("hive.metastore.uris", "thrift://hive-metastore:9083")
         .config("spark.sql.parquet.writeLegacyFormat", "true")   # de Hive 2.3 doc duoc
         .config("spark.driver.memory", "3g")
         .config("spark.sql.shuffle.partitions", "16")
         .enableHiveSupport().getOrCreate())
spark.sql("CREATE DATABASE IF NOT EXISTS fifa")
# Kiểm tra phiên bản Spark xem đã kết nối thành công chưa
print("Spark version:", spark.version)

ModuleNotFoundError: No module named 'pyspark'

## PHẦN 2: ĐỌC VÀ KIỂM TRA DỮ LIỆU TỪ HDFS
### Bước 1: Đọc dữ liệu cầu thủ từ HDFS vào Spark DataFrame

In [2]:
# Đọc file male_players.csv từ HDFS
df_male = spark.read.option("header", "true") \
                    .option("inferSchema", "true") \
                    .csv("hdfs://namenode:8020/fifa/raw/players/gender=male/male_players.csv")

# Kiểm tra số lượng dòng và hiển thị 5 dòng đầu tiên
print("Tổng số cầu thủ nam:", df_male.count())
df_male.show(5)

Tổng số cầu thủ nam: 10003590
+---------+--------------------+------------+-----------+----------------+--------------+--------------------+----------------+-------+---------+---------+--------+---+----------+---------+---------+---------+--------------+------------+------------+-------------------+-------------+------------------+----------------+----------------+------------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-------------+----------------+---------+------------------+--------------------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------

In [3]:
#Chuyển sang dạng xem theo chiều dọc (dễ nhìn toàn bộ chi tiết của 1-2 cầu thủ)
df_male.show(2, vertical=True, truncate=False)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------
 player_id                      | 158023                                                                                                         
 player_url                     | /player/158023/lionel-messi/230009                                                                             
 fifa_version                   | 23                                                                                                             
 fifa_update                    | 9                                                                                                              
 fifa_update_date               | 2023-01-13                                                                                                     
 short_name                     | L. Messi                                                                                  

### Bước 2: Kiểm tra cấu trúc, số lượng và dữ liệu mẫu

In [4]:
# Kiểm tra cấu trúc dữ liệu
df_male.printSchema()

# Kiểm tra các cột cần thiết cho ETL
columns_to_check = [
    "player_id", "short_name", "club_name",
    "nationality_name", "player_positions",
    "age", "overall", "potential",
    "wage_eur", "value_eur"
]

for c in columns_to_check:
    if c in df_male.columns:
        print(f"{c}: có trong dữ liệu")
    else:
        print(f"{c}: KHÔNG có trong dữ liệu")

root
 |-- player_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- league_id: integer (nullable = true)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = true)
 |-- club_team_id: integer (nullable = true)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nullable = true)
 |-- club_loane

## PHẦN 3: LÀM SẠCH DỮ LIỆU VÀ CHUẨN HÓA DỮ LIỆU BẰNG SPARK
### Bước 1: Kiểm tra chất lượng dữ liệu khi làm sạch
Kiểm tra kiểu dữ liệu, giá trị thiếu và các trường thông tin quan trọng để xác định những vấn đề cần xử lý trước khi thực hiện ETL.

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
# 1. Kiểm tra kiểu dữ liệu
df_male.printSchema()

# 2. Kiểm tra giá trị thiếu ở các cột quan trọng
key_cols = [
    c for c in [
        "player_id",
        "short_name",
        "club_name",
        "nationality_name",
        "player_positions",
        "age",
        "overall",
        "potential",
        "wage_eur",
        "value_eur"
    ]
    if c in df_male.columns
]
missing_stats = df_male.agg(*[
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in key_cols
])
print("Số lượng giá trị NULL ở các cột quan trọng:")
missing_stats.show(truncate=False)

root
 |-- player_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- league_id: integer (nullable = true)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = true)
 |-- club_team_id: integer (nullable = true)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nullable = true)
 |-- club_loane

### *Kết quả kiểm tra giá trị thiếu trong dữ liệu*

Sau khi thực hiện kiểm tra dữ liệu bằng Apache Spark, kết quả thống kê giá trị NULL trên 10 thuộc tính quan trọng của bộ dữ liệu cầu thủ nam FIFA cho thấy phần lớn các thuộc tính không có giá trị thiếu. Cụ thể, các cột `player_id`, `short_name`, `nationality_name`, `player_positions`, `age`, `overall` và `potential` đều có 0 giá trị NULL.

Tuy nhiên, một số thuộc tính vẫn tồn tại dữ liệu thiếu, bao gồm:

- **`club_name`:** có 116.395 giá trị NULL.
- **`wage_eur`:** có 115.974 giá trị NULL.
- **`value_eur`:** có 132.797 giá trị NULL.

Trong đó, `value_eur` là thuộc tính có số lượng giá trị thiếu lớn nhất trong các cột được kiểm tra. Những giá trị này có thể ảnh hưởng đến các phép thống kê về giá trị thị trường, tiền lương và phân tích theo câu lạc bộ nếu không được xem xé bằng Spark.

>**Kết luận:** Kết quả kiểm tra cho thấy dữ liệu có chất lượng tương đối tốt ở các thuộc tính định danh cầu thủ, quốc tịch, vị trí thi đấu, tuổi và các chỉ số năng lực. Tuy nhiên, vẫn tồn tại giá trị thiếu ở các thuộc tính liên quan đến câu lạc bộ và tài chính. Vì vậy, cần tiếp tục xử lý và chuẩn hóa dữ liệu ở bước tiếp theo trước khi thực hiện các phép tổng hợp, xây dựng bảng dữ liệu phụ và benchmark bằng Spark.

### Bước 2: Chuẩn hóa chuỗi và xử lý giá trị thiếu
Chuẩn hóa các cột kiểu chuỗi bằng cách loại bỏ khoảng trắng thừa ở đầu và cuối, đồng thời chuyển các chuỗi rỗng hoặc giá trị thiếu được biểu diễn bằng văn bản thành giá trị NULL thực sự.

In [6]:
# Tạo DataFrame mới, không sửa trực tiếp biến dữ liệu gốc
df_male_clean = df_male

# Xác định các cột có kiểu dữ liệu chuỗi
string_cols = [
    field.name
    for field in df_male_clean.schema.fields
    if isinstance(field.dataType, StringType)
]

print("Số cột kiểu chuỗi:", len(string_cols))

# Chuẩn hóa chuỗi trên từng cột
for c in string_cols:
    cleaned_value = F.trim(F.col(c))

    df_male_clean = df_male_clean.withColumn(
        c,
        F.when(
            cleaned_value.isNull()
            | (cleaned_value == "")
            | (F.lower(cleaned_value).isin(
                "null", "none", "n/a", "na"
            )),
            F.lit(None).cast("string")
        ).otherwise(cleaned_value)
    )

print("Đã hoàn thành bước chuẩn hóa chuỗi.")

Số cột kiểu chuỗi: 44
Đã hoàn thành bước chuẩn hóa chuỗi.


### *Đoạn code này xử lý các vấn đề sau:*
- " Paris Saint Germain " → "Paris Saint Germain".
- Chuỗi rỗng "" → NULL.
- Chuỗi "N/A" hoặc "null" → NULL.
Các giá trị thiếu vẫn được giữ lại dưới dạng NULL, chưa bị xóa hàng loạt. Đây là lựa chọn an toàn hơn với bộ dữ liệu FIFA, vì một cầu thủ có thể thiếu thông tin câu lạc bộ hoặc lương nhưng vẫn có các thông tin khác hữu ích cho phân tích.

### Bước 3: Kiểm tra và loại bỏ bản ghi trùng lặp hoàn toàn

Loại bỏ các bản ghi trùng lặp hoàn toàn trên toàn bộ các cột nhằm hạn chế dữ liệu lặp. Không loại bỏ bản ghi chỉ dựa trên `player_id`, vì một cầu thủ có thể xuất hiện ở nhiều phiên bản FIFA khác nha

Chọn dropDuplicates() trên toàn bộ dòng, không phải dropDuplicates(["player_id"]). Điều này giúp tránh xóa nhầm lịch sử của cùng một cầu thủ ở các phiên bản FIFA khác nhau.u.

In [7]:
# Loại bỏ các dòng trùng hoàn toàn
df_male_clean = df_male_clean.dropDuplicates()

print("Đã áp dụng loại bỏ bản ghi trùng lặp hoàn toàn.")

Đã áp dụng loại bỏ bản ghi trùng lặp hoàn toàn.


### Bước 4: Kiểm tra kết quả sau làm sạch

Kiểm tra số lượng bản ghi, cấu trúc dữ liệu và các giá trị thiếu sau khi chuẩn hóa. So sánh kết quả với dữ liệu ban đầu để đánh giá những thay đổi sau quá trình làm sạch.

In [ ]:
# 1. So sánh số dòng trước và sau làm sạch
rows_before = df_male.count()
rows_after = df_male_clean.count()

print(f"Số dòng trước làm sạch: {rows_before:,}")
print(f"Số dòng sau làm sạch:   {rows_after:,}")
print(f"Số dòng chênh lệch:      {rows_before - rows_after:,}")

# 2. Kiểm tra lại cấu trúc dữ liệu
df_male_clean.printSchema()

# 3. Kiểm tra dữ liệu sau làm sạch
df_male_clean.show(5, truncate=False)

# 4. Kiểm tra NULL sau làm sạch ở các cột quan trọng
df_male_clean.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in key_cols
]).show(truncate=False)

Số dòng trước làm sạch: 10,003,590
Số dòng sau làm sạch:   10,003,590
Số dòng chênh lệch:      0
root
 |-- player_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- league_id: integer (nullable = true)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = true)
 |-- club_team_id: integer (nullable = true)
 |-- club_name: string (nullable = true)
 |-- club_posi

## PHẦN 4: ETL VÀ XÂY DỰNG CÁC BẢNG DỮ LIỆU PHỤ

### Bước 1: Thực hiện các phép biến đổi ETL cơ bản trên Dataframe (df_male)
Bây giờ dữ liệu đã sẵn sàng trong biến df_male, bạn hãy tạo các ô code mới trong Jupyter để lọc, làm sạch và trích xuất thông tin theo yêu cầu của đồ án:
1. Lọc dữ liệu cầu thủ có chỉ số tiềm năng cao (Potential >= 85):

In [2]:
# Lọc các cầu thủ có tiềm năng từ 85 trở lên
top_potential_players = df_male_clean.filter(F("potential") >= 85)
print("Số lượng cầu thủ tiềm năng cao:", top_potential_players.count())
top_potential_players.select("short_name", "club_name", "overall", "potential").show(5)

NameError: name 'df_male_clean' is not defined

2. Tính toán thống kê nhóm (Group By): Ví dụ tính tuổi trung bình và tổng giá trị thị trường (value_eur) theo từng quốc gia (nationality_name):

In [ ]:
from pyspark.sql import functions as F

# 1. Tính toán thống kê nhóm và đặt tên alias
nationality_stats = df_male_clean.groupBy("nationality_name") \
    .agg(
        F.count("player_id").alias("total_players"),
        F.avg("overall").alias("avg_overall"),
        F.sum("value_eur").alias("total_value")
    ) \
    .orderBy(F.col("total_value").desc())  # Dùng F.col để sắp xếp theo cột vừa tạo

# 2. Hiển thị 10 quốc gia dẫn đầu
nationality_stats.show(10, truncate=False)

+----------------+-------------+-----------------+-------------+
|nationality_name|total_players|avg_overall      |total_value  |
+----------------+-------------+-----------------+-------------+
|Spain           |588413       |69.31521567334508|2389552440000|
|Brazil          |445413       |70.66080918159102|1842682000000|
|France          |543009       |67.14285582743564|1804694956000|
|Germany         |562575       |66.30997644758477|1539256940000|
|Argentina       |540117       |67.77604481991865|1453538656000|
|England         |931078       |62.92898017137125|1440395498000|
|Italy           |362619       |67.49908030191469|1041608264000|
|Portugal        |195911       |70.03397971527887|876514670000 |
|Netherlands     |243430       |67.55568335866573|732192425000 |
|Belgium         |156383       |67.74006125985561|608770690000 |
+----------------+-------------+-----------------+-------------+
only showing top 10 rows



### Bước 2: Tạo và lưu các bảng phụ (Auxiliary Tables)
Trong các yêu cầu của đồ án Big Data, việc tạo các bảng phụ tổng hợp (ví dụ: thống kê theo CLB, theo quốc tịch, hoặc theo vị trí) giúp tối ưu hóa truy vấn sau này. Bạn có thể lưu các bảng phụ này xuống định dạng Parquet (định dạng tối ưu nhất cho Spark) trên HDFS:

In [6]:
# BẢNG PHỤ 1: THỐNG KÊ THEO  QUỐC GIA X
nationality_stats.write.mode("overwrite").parquet("hdfs://namenode:8020/fifa/processed/nationality_summary")

print("Đã lưu bảng phụ thành công xuống HDFS!")

Đã lưu bảng phụ thành công xuống HDFS!


### Bước 3: Tối ưu hóa (Optimization) & Benchmark
Caching / Persisting: Khi bạn thực hiện nhiều thao tác lặp đi lặp lại trên DataFrame lớn (như df_male), hãy dùng lệnh cache để giữ nó trên bộ nhớ RAM, giúp các câu lệnh chạy nhanh hơn ở những lần gọi sau:

In [11]:
print(spark)

In [ ]:
# Lấy 5 dòng đầu tiên và hiển thị dưới dạng bảng trực quan
display(df_male_clean.limit(5).toPandas())

,player_id,player_url,fifa_version,fifa_update,fifa_update_date,short_name,long_name,player_positions,overall,potential,...,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk,player_face_url
0,158023,/player/158023/lionel-messi/230009,23,9,2023-01-13,L. Messi,Lionel Andrés Messi Cuccittini,RW,91,91,...,63+3,63+3,64+3,59+3,50+3,50+3,50+3,59+3,19+3,https://cdn.sofifa.net/players/158/023/23_120.png
1,165153,/player/165153/karim-benzema/230009,23,9,2023-01-13,K. Benzema,Karim Benzema,"CF, ST",91,91,...,64+3,64+3,64+3,60+3,55+3,55+3,55+3,60+3,18+3,https://cdn.sofifa.net/players/165/153/23_120.png
2,188545,/player/188545/robert-lewandowski/230009,23,9,2023-01-13,R. Lewandowski,Robert Lewandowski,ST,91,91,...,66+3,66+3,64+3,61+3,60+3,60+3,60+3,61+3,19+3,https://cdn.sofifa.net/players/188/545/23_120.png
3,192985,/player/192985/kevin-de-bruyne/230009,23,9,2023-01-13,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,...,79+3,79+3,78+3,74+3,68+3,68+3,68+3,74+3,21+3,https://cdn.sofifa.net/players/192/985/23_120.png
4,231747,/player/231747/kylian-mbappe/230009,23,9,2023-01-13,K. Mbappé,Kylian Mbappé Lottin,"ST, LW",91,95,...,63+3,63+3,67+3,63+3,54+3,54+3,54+3,63+3,18+3,https://cdn.sofifa.net/players/231/747/23_120.png


In [ ]:
# Benchmark thời gian chạy:
# Đo thời gian thực thi của truy vấn lớn bằng time.perf_counter().
# Kết quả giúp đánh giá hiệu suất xử lý dữ liệu của PySpark.
import time
from pyspark.sql.functions import avg

start = time.perf_counter()

df_male_clean.groupBy("club_name") \
    .agg(avg("wage_eur").alias("avg_wage")) \
    .show(10)

end = time.perf_counter()

print(f"Thời gian thực thi: {end - start:.2f} giây")

+---------------+------------------+
|      club_name|          avg_wage|
+---------------+------------------+
|    Real Madrid|139740.27214823393|
|        Antwerp| 9890.172212182417|
|AFC Bournemouth|26211.219663269327|
|     Bragantino|          12781.25|
|      Feyenoord|14628.237560192616|
|      Brentford|14885.671342685371|
|          Basel|15881.698033410868|
|  Werder Bremen| 17304.44029622376|
|       Al Hilal| 19194.30099033963|
|Vélez Sarsfield| 8160.742614452563|
+---------------+------------------+
only showing top 10 rows

Thời gian thực thi: 29.26 giây


### Bước 4: Tạo bảng phụ theo câu lạc bộ
Bảng này giúp trả lời những câu hỏi như:
- Câu lạc bộ nào có tổng giá trị thị trường cầu thủ cao?
- Câu lạc bộ nào có mức lương trung bình cao?
- Chất lượng cầu thủ trung bình giữa các câu lạc bộ khác nhau thế nào?

In [ ]:
# BẢNG PHỤ 2: Thống kê cầu thủ theo câu lạc bộ
club_stats = (
    df_male_clean
    .groupBy("club_name")
    .agg(
        F.count("player_id").alias("total_players"),
        F.avg("overall").alias("avg_overall"),
        F.avg("wage_eur").alias("avg_wage"),
        F.sum("value_eur").alias("total_value")
    )
    .orderBy(F.col("total_value").desc())
)

# Hiển thị 10 câu lạc bộ đầu tiên
club_stats.show(10, truncate=False)

+-------------------+-------------+-----------------+------------------+------------+
|club_name          |total_players|avg_overall      |avg_wage          |total_value |
+-------------------+-------------+-----------------+------------------+------------+
|Real Madrid        |17270        |77.97637521713955|139740.27214823393|442292395000|
|FC Barcelona       |17029        |78.58118503728933|140428.84784778906|424759840000|
|Manchester City    |18226        |75.6339295511906 |103879.9434873258 |412909850000|
|FC Bayern München  |15556        |78.34629724865003|97745.69298020056 |409758030000|
|Paris Saint Germain|15973        |77.21141927001815|78527.1927627872  |360945200000|
|Liverpool          |19729        |74.20898170206296|74018.9771402504  |345620595000|
|Chelsea            |18171        |75.45385504375103|88875.16372241484 |335641930000|
|Manchester United  |18351        |76.15367009972209|90580.22178627868 |326445265000|
|Juventus           |15181        |79.55753902904947|1

### Bước 5: Tạo bảng phụ theo vị trí thi đấu
Lưu ý: player_positions có thể chứa nhiều vị trí trong cùng một ô, chẳng hạn RW, ST. Khi đó, kết quả đang thống kê theo chuỗi vị trí được ghi trong dữ liệu, chưa tách riêng từng vị trí.

In [ ]:
# BẢNG PHỤ 3: Thống kê theo vị trí thi đấu

position_stats = (
    df_male_clean
    .groupBy("player_positions")
    .agg(
        F.count("player_id").alias("total_players"),
        F.avg("overall").alias("avg_overall"),
        F.avg("potential").alias("avg_potential"),
        F.avg("value_eur").alias("avg_value")
    )
    .orderBy(F.col("total_players").desc())
)

position_stats.show(10, truncate=False)

+----------------+-------------+------------------+-----------------+------------------+
|player_positions|total_players|avg_overall       |avg_potential    |avg_value         |
+----------------+-------------+------------------+-----------------+------------------+
|CB              |1244309      |66.43980554669298 |70.98372269267522|2272036.4716935926|
|GK              |1120906      |64.32804624116562 |69.2968125783964 |1657950.2372159888|
|ST              |1005942      |65.36765340347654 |70.53269273974047|2392515.10693027  |
|CM              |433166       |61.704889118721226|69.47148668178019|1645042.521898743 |
|CDM, CM         |404344       |68.37848960291237 |72.14822032724611|2746751.9894166053|
|LB              |385663       |65.13859509468111 |70.10936750479046|1675424.925362553 |
|RB              |342671       |65.16155437723064 |70.0435607331814 |1657049.2499793463|
|CM, CDM         |332653       |67.5802082049463  |71.67278214836482|3184618.87518629  |
|CDM             |199

### Bước 6: Lưu các bảng phụ lên HDFS dưới dạng Parquet
Lưu 2 bảng phụ theo câu lạc bộ và vị trí thi đấu

In [21]:
# Lưu bảng phụ theo câu lạc bộ
club_stats.write.mode("overwrite").parquet(
    "hdfs://namenode:8020/fifa/processed/club_summary"
)

# Lưu bảng phụ theo vị trí thi đấu
position_stats.write.mode("overwrite").parquet(
    "hdfs://namenode:8020/fifa/processed/position_summary"
)

print("Đã lưu các bảng phụ lên HDFS!")

Đã lưu các bảng phụ lên HDFS!


In [22]:
# Kiểm tra các thư mục đã lưu. Nếu kết quả đều là True, các thư mục tương ứng đang tồn tại trên HDFS.
spark._jsc.hadoopConfiguration().get("fs.defaultFS")

hdfs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jsc.hadoopConfiguration()
)

for path in [
    "/fifa/processed/nationality_summary",
    "/fifa/processed/club_summary",
    "/fifa/processed/position_summary"
]:
    print(path, "->", hdfs.exists(
        spark._jvm.org.apache.hadoop.fs.Path(path)
    ))

/fifa/processed/nationality_summary -> True
/fifa/processed/club_summary -> True
/fifa/processed/position_summary -> True


## PHẦN 5: TỐI ƯU HÓA BẰNG CACHE VÀ BENCHMARK
Hiện tại bạn đã đo một truy vấn mất khoảng 29,26 giây, nhưng cần so sánh thời gian chạy trước và sau khi cache để có cơ sở đánh giá.
### Bước 1: Đo thời gian truy vấn trước khi cache

In [ ]:
import time

# Benchmark trước khi cache
start = time.perf_counter()

df_male_clean.groupBy("club_name").agg(
    F.avg("wage_eur").alias("avg_wage")
).show(10)

time_before_cache = time.perf_counter() - start

print(f"Thời gian trước cache: {time_before_cache:.2f} giây")

+---------------+------------------+
|      club_name|          avg_wage|
+---------------+------------------+
|    Real Madrid|139740.27214823393|
|        Antwerp| 9890.172212182417|
|AFC Bournemouth|26211.219663269327|
|     Bragantino|          12781.25|
|      Feyenoord|14628.237560192616|
|      Brentford|14885.671342685371|
|          Basel|15881.698033410868|
|  Werder Bremen| 17304.44029622376|
|       Al Hilal| 19194.30099033963|
|Vélez Sarsfield| 8160.742614452563|
+---------------+------------------+
only showing top 10 rows

Thời gian trước cache: 29.55 giây


### Bước 2: Cache DataFrame và đo lại 
df_male.count() được dùng để kích hoạt việc tính toán và nạp DataFrame vào cache trước khi đo truy vấn. Vì vậy, thời gian nạp dữ liệu ban đầu không nằm trong thời gian benchmark thứ hai.

Lưu ý: Cache có thể làm truy vấn lặp lại nhanh hơn, nhưng không đảm bảo lúc nào cũng nhanh hơn. Kết quả còn phụ thuộc vào RAM, tài nguyên cụm, dữ liệu và tải hệ thống.

In [ ]:
# Lưu DataFrame vào bộ nhớ để tái sử dụng
df_male_clean.cache()

# Kích hoạt việc đọc và lưu dữ liệu vào cache
df_male_clean.count()

# Benchmark sau khi cache
start = time.perf_counter()

df_male_clean.groupBy("club_name").agg(
    F.avg("wage_eur").alias("avg_wage")
).show(10)

time_after_cache = time.perf_counter() - start

print(f"Thời gian sau cache: {time_after_cache:.2f} giây")

+---------------+------------------+
|      club_name|          avg_wage|
+---------------+------------------+
|    Real Madrid|139740.27214823393|
|        Antwerp| 9890.172212182417|
|AFC Bournemouth|26211.219663269327|
|     Bragantino|          12781.25|
|      Feyenoord|14628.237560192616|
|      Brentford|14885.671342685371|
|          Basel|15881.698033410868|
|  Werder Bremen| 17304.44029622376|
|       Al Hilal| 19194.30099033963|
|Vélez Sarsfield| 8160.742614452563|
+---------------+------------------+
only showing top 10 rows

Thời gian sau cache: 5.19 giây


### Bước 3: Tính mức cải thiện hiệu suất 
Nếu phần trăm dương, thời gian chạy giảm; nếu phần trăm âm, thời gian chạy tăng. Để kết quả đáng tin cậy hơn, bạn nên chạy mỗi trường hợp vài lần và so sánh các lần chạy tương ứng, vì lần chạy đầu có thể chịu ảnh hưởng của việc khởi tạo Spark và đọc dữ liệu từ HDFS.

In [ ]:
import pandas as pd
from IPython.display import display

# Tính mức cải thiện hiệu suất
if time_before_cache > 0:
    improvement = (
        (time_before_cache - time_after_cache)
        / time_before_cache
    ) * 100

    # Tạo bảng kết quả benchmark
    benchmark_results = pd.DataFrame({
        "Chỉ tiêu": [
            "Thời gian trước khi cache",
            "Thời gian sau khi cache",
            "Mức cải thiện hiệu suất"
        ],
        "Giá trị": [
            f"{time_before_cache:.2f} giây",
            f"{time_after_cache:.2f} giây",
            f"{improvement:.2f}%"
        ]
    })

    display(benchmark_results)

# Giải phóng cache, tránh hiển thị DataFrame dài
_ = df_male_clean.unpersist()

print("Đã giải phóng cache thành công.")

,Chỉ tiêu,Giá trị
0,Thời gian trước khi cache,29.55 giây
1,Thời gian sau khi cache,5.19 giây
2,Mức cải thiện hiệu suất,82.43%


Đã giải phóng cache thành công.


## PHẦN 6: KIỂM TRA KẾT QUẢ VÀ TỔNG KẾT

In [27]:
# Kiểm tra đọc lại 3 bảng phụ từ HDFS

nationality_check = spark.read.parquet(
    "hdfs://namenode:8020/fifa/processed/nationality_summary"
)

club_check = spark.read.parquet(
    "hdfs://namenode:8020/fifa/processed/club_summary"
)

position_check = spark.read.parquet(
    "hdfs://namenode:8020/fifa/processed/position_summary"
)

# Hiển thị số dòng của từng bảng
print("Số dòng bảng quốc tịch:", nationality_check.count())
print("Số dòng bảng câu lạc bộ:", club_check.count())
print("Số dòng bảng vị trí:", position_check.count())

Số dòng bảng quốc tịch: 193
Số dòng bảng câu lạc bộ: 1649
Số dòng bảng vị trí: 2905


# TỔNG KẾT
- Nhiệm vụ: Xử lý dữ liệu bằng Apache Spark, xây dựng các bảng dữ liệu phụ, tối ưu hóa quá trình xử lý và đánh giá hiệu suất thực thi truy vấn.
- Đầu tiên, thiết lập kết nối giữa Spark và HDFS, sau đó đọc dữ liệu cầu thủ nam từ tệp male_players.csv vào Spark DataFrame. Bộ dữ liệu được sử dụng trong quá trình xử lý gồm 10.003.590 dòng. Trên cơ sở đó, thực hiện các thao tác ETL cơ bản như lọc cầu thủ có chỉ số tiềm năng từ 85 trở lên, thống kê số lượng cầu thủ và tổng hợp các chỉ số phục vụ phân tích.

- Tiếp theo, xây dựng ba bảng dữ liệu phụ theo quốc tịch, câu lạc bộ và vị trí thi đấu. Các bảng cung cấp thông tin tổng hợp như số lượng cầu thủ, chỉ số tổng thể trung bình, mức lương trung bình, chỉ số tiềm năng trung bình và giá trị thị trường, tùy theo từng nhóm dữ liệu. Sau khi hoàn thành, các bảng phụ được lưu trên HDFS dưới định dạng Parquet. Các thư mục lưu trữ đã được kiểm tra và xác nhận tồn tại, nhằm bảo đảm dữ liệu đã được ghi vào hệ thống lưu trữ phân tán.

- Đối với việc tối ưu hiệu suất, sử dụng cơ chế cache của Spark để lưu DataFrame vào bộ nhớ khi thực hiện các thao tác lặp lại. Tiến hành benchmark truy vấn tính mức lương trung bình theo câu lạc bộ trước và sau khi cache, đồng thời tính toán mức thay đổi thời gian thực thi. Kết quả của lần đo thử nghiệm ghi nhận mức thay đổi thời gian là 82,43%. Kết quả này được sử dụng để đánh giá tác động của cache đối với truy vấn trong điều kiện thực nghiệm đã thực hiện. Sau khi hoàn thành benchmark, giải phóng cache bằng lệnh unpersist() để tránh chiếm dụng bộ nhớ không cần thiết.

> *Nhìn chung, nhiệm vụ Spark ETL đã thực hiện được các nội dung chính gồm đọc và xử lý dữ liệu, tạo các bảng phụ phục vụ truy vấn phân tích, lưu trữ dữ liệu tổng hợp dưới định dạng Parquet trên HDFS và thực nghiệm đánh giá hiệu suất thông qua benchmark. Kết quả đạt được góp phần chuẩn bị dữ liệu có cấu trúc cho các bước khai thác và phân tích tiếp theo của dự án Big Data.*